In [13]:
import pandas as pd
import geopandas as gpd

county = "harris"
fips = "201"

# --- Load census tracts and filter to county ---
tracts = gpd.read_file("../../Census_Tract/tl_2025_48_tract.shp")
county_tracts = tracts[tracts["COUNTYFP"] == fips]
print(f"Tracts in county: {len(county_tracts)}")

# --- Dissolve tracts into one county boundary polygon ---
county_boundary = county_tracts.dissolve().to_crs("EPSG:4326")

# --- Load your farms ---
df = pd.read_csv(f"../deduplicated_farms/completed_deduplicated_data/deduplicated_{county}_farms.csv")
print(f"Farms before filter: {len(df)}")

# --- Convert to GeoDataFrame ---
gdf = gpd.GeoDataFrame(
    df,
    geometry=gpd.points_from_xy(df["lon"], df["lat"]),
    crs="EPSG:4326"
)

# --- Spatial filter ---
filtered = gdf[gdf.geometry.within(county_boundary.union_all())]
print(f"Farms after filter: {len(filtered)}  ({len(df) - len(filtered)} outside county removed)")

# --- Save ---
filtered.drop(columns="geometry").to_csv(f"deduplicated_{county}_farms.csv", index=False)
print(f"Saved to deduplicated_{county}_farms.csv")

Tracts in county: 1115
Farms before filter: 204
Farms after filter: 116  (88 outside county removed)
💾 Saved to deduplicated_harris_farms.csv
